<a href="https://colab.research.google.com/github/khadearti1102-SP/AgenticAI_Report_Writing_Assistant_v1.0/blob/main/Research_Assistant_SBS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install langgraph langchain  langchain-groq langchain-community tavily-python langsmith python-dotenv pydantic tavily

In [13]:
from typing_extensions import TypedDict, Annotated, Dict
from operator import add
from google.colab import userdata
import os
from langchain_groq import ChatGroq
from tavily import TavilyClient
from langchain_community.document_loaders import WebBaseLoader

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
def get_llm(temperature: 0.2) -> ChatGroq:
  return ChatGroq(
      model="llama-3.3-70b-versatile",
      temperature=0,
      max_retries=2,
      timeout=60
  )

llm = get_llm(temperature=0.2)
# response = llm.invoke("Kitne aadmi the?")
# print(f"Response: {response.content}")

class PlanState(TypedDict):
  reasoning: str
  questions: list[str]

class Citation(TypedDict):
  claim: str
  source_url: str

class ResearchState(TypedDict):
  topic: str
  plans: list[PlanState]
  research_plan: str
  plan_reasoning: str
  raw_findings: list[dict]
  total_steps_taken: int
  analysis: str
  conflicts: list[str]
  draft_report: str
  citations: list[Citation]
  context_segment: str

class Finding(TypedDict):
  query: str
  source: str
  url: str
  content: str
  relevance_note: str

def parse_json(text: str) -> dict:
  """Strip markdown fences if the model added them, then parse"""
  cleaned = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
  return json.loads(cleaned)

#TOOLS: SEARCH TOOL
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
client = None

def get_client() -> TavilyClient:
  global client
  if client is None:
    client = TavilyClient()
  return client


BLOCKED_DOMAINS = {    "reddit.com", "quora.com", "medium.com", "blogspot.com" "wordpress.com", "x.com", "twitter.com", "facebook.com", "instagram.com", "linkedin.com", "slideshare.net", "ezinearticles.com"}

def web_search(query: str, max_results: int = 3) -> list[dict]:
  """
  Run a web search and return cleaned results.
  Returns list of dictionaries: {title, url, content}
  """
  #print(f"Searching for query: {query}")
  client = get_client()
  try:
    response = client.search(
        query,
        max_results=max_results,
        search_depth="advanced"
    )
    #print(f"Response: {response}")
  except Exception as e:
    #print(f"Error: {e}")
    return []

  results = []
  for r in response.get("results", []):
    url = r.get("url", "")
    if any (blocked in url for blocked in BLOCKED_DOMAINS):
      continue
    results.append({
        "title": r.get("title", ""),
        "url": r.get("url", ""),
        "content": r.get("snippet", "")
    })
    #print(f"Found {len(results)} results")
  return results


def fetch_url_content(url: str) -> str:
  """
  Fetch and return raw text content of URL for deeper reading.
  """
  loader = WebBaseLoader(url)
  data = loader.load()
  print("\n data fetched")
  return "\n".join(p.page_contnet for p in data)[:5000]

PLANNER NODE


*   Planning
*   Evaluate the plan and finalize one



In [14]:
from typing import TypedDict, Dict
import json
PLANS = 3
BRANCH_PROMPT = """
You are a research planning assistant. Your job is to break down a topic into research strategies
Topic to research: {topic}
{context_segment}
ONLY if context_segment is non empty: CRITICAL RULE: At least one of your proposed
research plans MUST actively generate questions designed to investigate,
cross-reference, or validate the specific claims made in the case study context
provided above against broader web data.

Propose exactly three distinct research plans. Each plan must address the topic
from a completely different angle/approach
-- vary between angles like: current
state/data, competing viewpoints, future implications, practical
applications, risks/criticisms, historical context.
Each plan must contain a list of 3-5 questions that, if answered, would let
someone write a thorough, well sourced report on this topic.

Return only JSON:
{{ "plans":
[{{
"reasoning": "<Justification for why this angle is valuable for understanding the topic>",
"questions": [ "<question1>", "<question2>", "<question3>" ]
}}]
 }}
  """

EVAL_PROMPT = """
You are an evaluating research plans for the topic: {topic}
Here are {n} proposed plans: {plans}
Score each plan (1-10 for coverage, non-redundancy, report quality). Choose the
best plan or hybrid.
Respond ONLY with JSON:
{{"reasoning": "<brief comparison of the plans>",
"chosen_sub_questions: ["....", "...."]"}}
"""

def parse_json(text: str) -> dict:
  """Strip markdown fences if the model added them, then parse"""
  cleaned = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
  return json.loads(cleaned)

def planner_node(state: ResearchState) -> json:
  topic = state["topic"]
  context = state.get("user_context", "").strip()

  plans = []
  context_segment = ""
  if context:
    context_segment = (
            f"Consider this context as absolute facts already discovered in our case study: {context}\n"
            f"Identify gaps or areas where external web data, broader market trends are needed. Do not search for what we already know."
            )
  resp = llm.invoke(BRANCH_PROMPT.format(topic=topic, context_segment=context_segment))
  parsed_resp = parse_json(resp.content)
  #print(parsed_resp)

  # handling if generation fails completely
  if not parsed_resp or "plans" not in parsed_resp:
    return {
        "plans": ["No plans generated"]
    }

  # Build the clean, nested JSON array
  plan_index = 1
  validated_plans = []
  for p in parsed_resp["plans"]:
    plan_state: PlanState ={
        "reasoning": p.get("reasoning", "No reasoning provided."),
        "questions": p.get("questions", [])
    }
    print(f"================Plan{plan_index}==============")
    print(plan_state)
    plan_index += 1
    validated_plans.append(plan_state)

  # Evaluate plans
  eval_prompt = EVAL_PROMPT.format(topic=topic, n=len(validated_plans), plans=json.dumps(validated_plans))
  eval_resp = llm.invoke(eval_prompt)

  try:
    evaluation = parse_json(eval_resp.content)
    chosen = evaluation.get("chosen_sub_questions", [])
    rationale = evaluation.get("reasoning", "No reasoning provided")
  except json.JSONDecodeError:
      chosen = plans[0].get("questions", [])
      rationale = "Evaluation step failed to parse --defaulted to first plan."
  # Return the nested JSON structure directly inside the state dictionary
  return {
      "research_plan": chosen,
      "plan_reasoning": rationale
  }

In [15]:
#RESEARCHER NODE
import json
from typing import Dict

MAX_REFINEMENTS_PER_QUESTION = 1

THINK_PROMPT = """
You are researching this question: "{question}"
{history}
Primary Case Study Context: {user_context}

Decide the best search query to run right now.
If this is a follow-up(history is non empty),
make the new query MORE specific than before,
targeting whatever was missing.

IF Case Study Context(user_context is non empty) is present, do NOT search for facts already stated there.
Instead, search for information that verifies, challenges, or fills gaps around
those facts.

Respond ONLY with JSON:
{{"query": "<query to run>",
  "reasoning": "....."}}
"""

OBSERVE_PROMPT = """Sub-question: "{question}"
Search query used: "{query}"

Results: {results}

Decide:
1. Which results (if any) are relevant and worth keeping as findings?
2. Is there enough information to move on, or does this need a follow-up
search with refined query?

Respond ONLY with JSON:
{{"Kept_findings": [{{"source": "...", "url": "...", "content": "...",
"relevance_note": "..."}}],
"sufficient": true/false}}
"""

def researcher_node(state: ResearchState) -> Dict:
  findings = []
  total_steps = 0
  user_context = state.get("context_segment", "")
  for question in state["research_plan"]:
    history = []
    for refinement in range(MAX_REFINEMENTS_PER_QUESTION):
      if history:
        history_str = ("Previous queries: \n" + "\n".join(history))
      else:
        history_str = ""

      think_prompt = THINK_PROMPT.format(question=question, history=history_str, user_context=user_context)
      think_resp = llm.invoke(think_prompt)
      # print({json.dumps(parse_json(think_resp), indent=2)})

      try:
        thought = parse_json(think_resp.content)
        #print(f"Thoughts: {thought}")
        query = thought.get("query", "")
        #print(f"query: {query}")
        thought_reasoning = thought.get("reasoning", "")
      except Exception as e:
        query = question

      try:
        results = web_search(query)
      except Exception as e:
        history.append(f"Query: {query}\nError: {e}")
        continue

      total_steps += 1
      if not results:
        history.append(f"Query: {query}\nError: No results found")
        continue

      # OBSERVE
      results_str = "\n".join(f"{i+1}. {r['title']}: {r['url']}" for i, r in enumerate(results))
      #print(f"Results: {results_str}")
      observe_prompt = OBSERVE_PROMPT.format(question=question, query=query, results=results_str)
      observe_resp = llm.invoke(observe_prompt)

      try:
        observation = parse_json(observe_resp.content)
        #print(observation)
        kept_findings = observation.get("Kept_findings", [])
        sufficient = observation.get("sufficient", False)
      except JSONDecodeError:
        kept_findings = []
        sufficient = False

      for finding in kept_findings:
        findings.append(
            Finding(
                query=query,
                source=finding.get("source", ""),
                url=finding.get("url", ""),
                content=finding.get("content", ""),
                relevance_note=finding.get("relevance_note", "")
            )
        )

        history.append(f"Query '{query}' -> kept {len(kept_findings)} findings")

        if sufficient:
          break

  return {
      "raw_findings": findings,
      "total_steps_taken": state.get("total_steps_taken", 0) + total_steps
  }



In [16]:
# ANALYST NODE
import json
from typing import Dict

ANALYST_PROMPT ="""
You are analyzing research findings on: {topic}
Research plan(questions investigated): {plan}
Findings tagged by source ID: {sources}
User context: {user_context}

Format as:
1. Narrative per question, citing[S#]
2. Do not invent, explicitly note insufficient findings.
3. Identify CONFLICTS: List contradictions between web sources, OR between the [Context Verification Required] fact
and web sources [S#]. State "None" if aligned.

Respond ONLY with JSON:
{{"analysisi": "<the narrative with [S#] tags inline>",
"conflicts": ["<conflict1, naming the source IDs involved>", "..."]}}
"""

def build_source_index(findings: list[Finding],user_context: str) -> Dict[str, list[Finding]]:
  """
  Same [S#] tagging scheme the writer uses, so IDs line up.
  """
  lines = []
  if user_context:
    lines.append(f"[S1] Internal User Context: {user_context}\n")
  for i, f in enumerate(findings, start=2):
    lines.append(
        f"[S{i}] Source: {f.get('source', 'unknown')} ({f.get("url", "no url")})\n"
        f"Content: {f.get('content', '')[:400]}\n"
    )
  #print(lines)
  return "\n".join(lines)

def analyst_node(state: ResearchState) -> Dict:
  topic = state["topic"]
  findings = state.get("raw_findings", [])
  user_context = state.get("context_segment", "")


  if not findings:
    return {
        "analysis": "No findings to analyze.",
        "conflicts": []
    }
  sources = build_source_index(state["raw_findings"], user_context=user_context)

  plan = "\n".join(f" - {q}" for q in state.get("research_plan", []))

  response = llm.invoke(ANALYST_PROMPT.format(topic=topic,
                                              plan=plan,
                                              sources=sources,
                                              user_context=user_context))

  try:
    result = parse_json(response.content)
    analysis = result.get("analysis", "")
    conflicts = result.get("conflicts", [])
  except json.JSONDecodeError:
    analysis = "Analysis failed to parse."
    conflicts = []

  return {
      "analysis": analysis,
      "conflicts": conflicts
  }





In [17]:
from collections import UserDict
# WRITER NODE
import re

WRITER_PROMPT = """You are an elite research report author. Your task is to extract and synthesize maximum detailed information from the provided repository into an exhaustive, data-dense markdown report.: {topic}

Research_plan (sub-questions investigated):
{plan}

Analyst's analysis (use this as your main guide for structure and
content -- it already has resolved conflicts between sources)
{analysis}

Known conflicts between sources are flagged by the Analyst (acknowledge these in the
report rather than silently picking one side):
{conflicts}

Available sourced findings(cite these using their [S#] tag inline whenever
you use them -- every factual claim must have a tag):
{sources}

Generate a well-structured markdown report:
- Address each sub-question.
- Cite all factual claims with [S#] tags.
- State if findings are incomplete for a sub-question.
- Do not invent information; use only provided sources.
- Acknowledge flagged conflicts.
- Include a "Summary" section.

Respond with the report text only -- no JSON, no preamble.
"""

def get_source_index(findings: list[dict], user_context: str) -> tuple[str, dict[str, dict]]:
  """
  Assign S2, S2... IDs to findings and build the text block shown
  to the LLM, plus a lookup dictionary for citation validation later
  """
  lines = []
  print(findings)
  index: dict[str, dict] = {}

  if user_context:
    sid = "S1"
    index[sid] = {
        "source": "Internal User Context",
        "url": "Internal Case Study",
        "content": user_context
    }
    lines.append(f"[S1] Internal User Context: {user_context}\n")
  for i, f in enumerate(findings, start=2):
    sid = f"S{i}"
    #print(sid)
    index[sid] = f
    lines.append(
        f"{sid} Source: {f.get('source', 'unknown')}"
        f"({f.get('url', 'no url')})\n"
        f"Content: {f.get('content', '')[:300]}\n"
    )
  #print(lines)
  return "\n".join(lines), index


def writer_node(state: ResearchState) -> dict:
  findings = state.get("raw_findings", [])
  user_context = state.get("context_segment", "")

  if not findings:
    return {
        "draft_report": "No findings to write a report on",
        "citations": []
    }

  sources_str, source_index = get_source_index(findings, user_context)
  plan_str = "\n".join(f" - {q}" for q in state.get("research_plan", []))
  analysis = state.get("analysis", "")
  conflicts = state.get("open_conflicts", [])
  conflicts_str = "\n".join(f" - {c}" for c in conflicts) if conflicts else "None flagged"

  response = llm.invoke(WRITER_PROMPT.format(
      topic=state.get("topic", ""),
      plan=plan_str,
      analysis=analysis,
      conflicts=conflicts_str,
      sources=sources_str
  ))

  draft = response.content
  used_tags = sorted(set(re.findall(r"\[S\d+\]", draft)))
  citations: list[Citation] = []
  for tag in used_tags:
    sid = tag.strip("[]")
    source = source_index.get(sid, {})
    if source:
      citations.append(
          Citation(
              claim=f"Claims tagged {tag}",
              source_url=source.get("url", ""),
          )
      )

  return {
      "draft_report": draft,
      "citations": citations
  }

In [18]:
from langgraph.graph import StateGraph, END, START


input = {
    "topic": "Data privacy risks of employees using public LLMs",
    "plans": [],
    "context_segment": ""
}

def build_graph():
  graph = StateGraph(ResearchState)
  graph.add_node("planner", planner_node)
  graph.add_node("researcher", researcher_node)
  graph.add_node("analyst", analyst_node)
  graph.add_node("writer", writer_node)
  graph.add_edge(START, "planner")
  graph.add_edge("planner", "researcher")
  graph.add_edge("researcher", "analyst")
  graph.add_edge("analyst", "writer")
  graph.add_edge("writer", END)
  return graph.compile()

if __name__ == "__main__":
  app = build_graph()

  final_state = app.invoke(input)

  print("\n======RESEARCH PLAN=========")
  for step in final_state["research_plan"]:
    print(step)

  print(f"\n=========PLAN REASONING=======")
  print(f"\n{final_state['plan_reasoning']}")

  print(f"\n=========FINDINGS=======")
  print(f"Found {len(final_state['raw_findings'])} findings:{final_state['raw_findings']}")
  # for finding in final_state["raw_findings"]:
  #   print(f"Query: {finding['query']}")
  #   print(f"Source: {finding['source']}")

  print(f"=====Total Steps Taken==========")
  print(f"\n{final_state['total_steps_taken']}")

  print(f"==========ANALYSIS==========")
  print(f"\n{final_state['analysis']}")

  print(f"==========CONFLICTS==========")
  print(f"\n{final_state['conflicts']}")

  print(f"==========DRAFT REPORT==========")
  print(f"\n{final_state['draft_report']}")

  print(f"\n=== CITATIONS ({len(final_state['citations'])}) ===")
  for c in final_state["citations"]:
      print(json.dumps(c, indent=2))



{'plans': [{'reasoning': 'Understanding the current state of data privacy risks associated with public LLMs is crucial for identifying existing vulnerabilities and potential threats. This approach will provide insights into the types of data that are being exposed, the frequency of data breaches, and the current measures being taken to mitigate these risks.', 'questions': ['What types of sensitive data are employees most likely to expose when using public LLMs?', 'How often do data breaches occur due to the use of public LLMs in the workplace?', 'What are the current best practices for mitigating data privacy risks associated with public LLMs?']}, {'reasoning': 'Examining competing viewpoints on the use of public LLMs in the workplace can provide a more nuanced understanding of the data privacy risks involved. This approach will consider the perspectives of different stakeholders, including employees, employers, and regulatory bodies.', 'questions': ['How do employees perceive the data